#### Topic classification and paragraph splitting with LLM 
- as of July 24, there is a bug with vllm 0.52 , have to build from source . follow: https://docs.vllm.ai/en/latest/getting_started/installation.html#build-from-source
- or just use vllm 0.51, but it won't work with gemma 2
- reference https://github.com/vllm-project/vllm/issues/6462
- also go to run gemma 2 , need to install ; pip install flashinfer -i https://flashinfer.ai/whl/cu121/torch2.3

In [49]:
import os,sys
sys.path.insert(0,'../libs')
sys.path.insert(0,'../src')
from utils import load_json
import openai
from llm_utils import BSAgent
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm
from huggingface_hub import hf_hub_download
from huggingface_hub import snapshot_download
from prompts import topic_identify_simple_pt, output_fixing_pt


ImportError: cannot import name 'output_fixing_pt' from 'prompts' (/root/dev/Fund_ISR/notebooks/../src/prompts.py)

In [84]:
import pprint

In [ ]:
from typing import Optional,List,Literal,Iterable
from langchain.output_parsers import PydanticOutputParser
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_openai import ChatOpenAI

#### Download model from model hub

In [2]:
def donload_hf_model(REPO_ID,save_location):
    # REPO_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
    # save_location = '/root/data/hf_cache/llama-3-8B-Instruct'
    hf_token = input("huggingface token:")
    snapshot_download(repo_id=REPO_ID,
                    local_dir=save_location,
                    token=hf_token)
    
    return save_location

In [5]:
# REPO_ID = "Qwen/Qwen2-7B-Instruct"
# # REPO_ID = "google/gemma-2-9b-it" 
# REPO_ID = "google/gemma-2-27b-it"
# REPO_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
# REPO_ID = "Qwen/Qwen2-72B-Instruct"
# REPO_ID = "meta-llama/Meta-Llama-3.1-70B"
# save_location = os.path.join('/root/data/hf_cache',REPO_ID)
# donload_hf_model(REPO_ID,save_location)

Fetching 50 files: 100%|██████████| 50/50 [06:40<00:00,  8.00s/it]


'/root/data/hf_cache/meta-llama/Meta-Llama-3.1-70B'

#### user vllm to serve api endpoint

- #### serve opensource model as openai end point
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/Qwen/Qwen2-7B-Instruct --dtype auto --api-key token-abc123 --port 8000
- CUDA_VISIBLE_DEVICES=4,5,6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/Qwen/Qwen2-72B-Instruct --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 4
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/meta-llama/Meta-Llama-3.1-8B-Instruct --dtype auto --api-key token-abc123 --port 8000
- CUDA_VISIBLE_DEVICES=4,5,6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/meta-llama/Meta-Llama-3.1-70B-Instruct --dtype auto --api-key token-abc123 --port 8000 --tensor-parallel-size 4
- export VLLM_ATTENTION_BACKEND=FLASHINFER
- CUDA_VISIBLE_DEVICES=6,7  python -m vllm.entrypoints.openai.api_server --model /root/data/hf_cache/google/gemma-2-27b-it --dtype auto --api-key token-abc123 --port 8000 ----pipeline-parallel-size 2

In [13]:
# Modify OpenAI's API key and API base to use vLLM's API server.
openai_api_key = "token-abc123"
openai_api_base = "http://localhost:8000/v1"

llm_agent  = BSAgent(api_key=openai_api_key,
                     api_base_url=openai_api_base,
                    temperature=0)
llm_agent.model = llm_agent.client.models.list().data[0].id
print('current running model :{}'.format(llm_agent.model))

current running model :/root/data/hf_cache/meta-llama/Meta-Llama-3.1-8B-Instruct


In [ ]:
error_fixing_llm = ChatOpenAI(openai_api_base=openai_api_base,
                              openai_api_key=openai_api_key)

#### Try one example

In [14]:
## just run one test, make sure the api works 
pt = {'System':'You are a helpful assistant.',
      'Human':'Who won the world series in 2020?'}
if 'gemma' in llm_agent.model:
    pt = {'Human':'Who won the world series in 2020?'}  
res = llm_agent.get_response_content(prompt_template=pt, 
                                     #max_tokens=4096,
                                     temperature=0)
print(res)  

The Los Angeles Dodgers won the 2020 World Series. They defeated the Tampa Bay Rays in the series 4 games to 2. This was the Dodgers' first World Series title since 1988.


In [22]:
class Topic(BaseModel):
    topic_label: Literal["Economic Outlook", "Monetary Policy", "Fiscal Stance", 
                         "Financial Stability","External Stance","Climate Change",
                         "Inclusion","Technology","Governance","Structural Reforms",
                         "Other Topics"] = Field(description="short title for the topic")
    confidence_score: int = Field(description="confidence score of topic identification, ranges from 0 to 100")

class Topic_Result(BaseModel):
    reasoning: str = Field(description="the reasoning process for topic identification")
    topic_labels: List[Topic] = Field(description="list of topics that are identified")
    
parser = PydanticOutputParser(pydantic_object=Topic_Result)

#pprint.pprint(parser.get_format_instructions())
example_output = '{"reasoning": "test test test", "topic_labels": [{"topic_label":"Other Topics","confidence_score":10}]}'
parser.parse(example_output)

Topic_Result(reasoning='test test test', topic_labels=[Topic(topic_label='Other Topics', confidence_score=10)])

#### Try one paragraph and see how well it works 

In [10]:
df = pd.read_csv('/root/data/Fund/CSR/detailed_topic_identification_eval_small.csv')
df.head()

,paragraph,topic_model_label,mapped_topic_label,ground_truth_label,Notes
0,"Additional, targeted fiscal support is warrant...",Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
1,The authorities agreed that additional fiscal ...,Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
2,Box 1. Jordan: Past Fund Staff Advice and Impl...,Fiscal Management,Fiscal Stance,Fiscal Stance,NaN
3,The authorities consider that geopolitical ten...,Financial Stability,Financial Stability,Financial Stability,NaN
4,The banking sector is in good financial health...,Financial Stability,Financial Stability,Financial Stability,NaN


In [29]:
df.rename(columns={"paragraph": "examples"}, inplace=True)
print(df.iloc[2]['ground_truth_label'])
print(df.iloc[2]['examples'])
test_input = df.iloc[2]

Fiscal Stance
Box 1. Jordan: Past Fund Staff Advice and Implementation Macroeconomic policies were implemented with delays but broadly in line with past staff advice. Despite efforts, however, fiscal consolidation fell short of expectations, reflecting sizable revenue shortfalls— due to widespread weaknesses in tax administration, low-quality measures, delays in implementation, and some tax policy reversals—and public financial management deficiencies.
Improving the tax framework. Macro-critical measures, such as the reduction of sales-tax exemptions and amendments to the income tax law, have strengthened the tax policy framework. However, persistent weaknesses in tax administration, inability to tackle tax evasion and smuggling, and some tax policy reversals contributed to sizable revenue slippages over 2018-19. To reduce deviations from program targets, the authorities have often relied on low-quality measures, including cuts to capital spending and postponing the clearance of domest

#### Run one basic economic sector question and see if LLM knows basic macro

In [30]:
if 'gemma' in llm_agent.model:
    topic_identify_simple_pt['Human']  = topic_identify_simple_pt.get('System') + topic_identify_simple_pt.get('Human') 
    del topic_identify_simple_pt['System']

In [32]:
#print(topic_identify_simple_pt['Human'].format(PARA=test_input['examples']))
topic_pt_temp = copy.copy(topic_identify_simple_pt)
topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=test_input['examples'])
res = llm_agent.get_response_content(prompt_template=topic_pt_temp,max_tokens=4096)


In [33]:
## two different ways of parsing results 
res_dict = llm_agent.parse_load_json_str(res) 
print(res_dict)
res_dict2 = parser.parse(res)
print(res_dict2)

{'reasoning': "The paragraph discusses various economic policies and reforms implemented in Jordan, including fiscal consolidation, tax administration, public financial management, and structural reforms. It also touches on monetary policy, exchange-rate peg, and social safety net. The discussion is focused on the country's economic performance and the implementation of IMF staff advice. The topics of climate change, inclusion, technology, and governance are not explicitly mentioned, but some of the reforms, such as reducing payroll taxes, could be related to inclusion. However, the primary focus is on economic policies and reforms, making it difficult to assign multiple topics. Therefore, I will assign the topic that best fits the overall content of the paragraph.", 'topic_labels': [{'topic_label': 'Fiscal Stance', 'confidence_score': 80}]}
reasoning="The paragraph discusses various economic policies and reforms implemented in Jordan, including fiscal consolidation, tax administration

In [34]:
sample_results = []
for idx,r in tqdm(df.iterrows()):
    topic_pt_temp = copy.copy(topic_identify_simple_pt)
    topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=r['examples'])
    res = llm_agent.get_response_content(prompt_template=topic_pt_temp,max_tokens=4096)
    try:
        #res_dict = llm_agent.parse_load_json_str(res) 
        #sample_results.append([r['llm_topic'],r['examples'],res_dict['reasoning'],res_dict['topic_labels']])
        res_dict = parser.parse(res).dict()
        sample_results.append([r['ground_truth_label'],r['examples'],res,res_dict['reasoning'],res_dict['topic_labels']])
    except:
        sample_results.append([r['ground_truth_label'],r['examples'],res,None,None])

100it [19:32, 11.72s/it]


In [36]:
sample_results[0]

['Fiscal Stance',
 'Additional, targeted fiscal support is warranted in 2022-2023. The 2022 Budget allocated RM23 billion (1.4 percent of GDP) out of the COVID Fund for COVID-related spending, with an increasing share spent on upskilling programs, social assistance support to vulnerable groups, and SME financing. Considering the still-sizable projected economic slack, medium-term pandemic scarring effects, generally favorable debt dynamics, and staff’s assessment that debt is sustainable (Appendix II), staff advice is to provide modest additional fiscal support by about 2 percentage points cumulatively in 2022 and 2023 above the 2022 budget plans, moving the fiscal impulse in 2022 from negative to moderately positive territory, and delaying the start of consolidation by one year to 2023.8 The additional stimulus would close the output gap faster than in the baseline, thus helping limit the pandemic’s scarring effects. Targeted additional spending should center on protecting the vulnera

In [150]:
res_df = pd.DataFrame(sample_results,columns=['ground_truth_label','paragraph','llm_raw_response','reasoning','topic_labels'])
res_df.head(10)

,ground_truth_label,paragraph,llm_raw_response,reasoning,topic_labels
0,Fiscal Stance,"Additional, targeted fiscal support is warrant...","```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the economic outlook a...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."
1,Fiscal Stance,The authorities agreed that additional fiscal ...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the fiscal policy and ...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."
2,Fiscal Stance,Box 1. Jordan: Past Fund Staff Advice and Impl...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses various economic polic...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."
3,Financial Stability,The authorities consider that geopolitical ten...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the financial stabilit...,"[{'topic_label': 'Financial Stability', 'confi..."
4,Financial Stability,The banking sector is in good financial health...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the banking sector's f...,"[{'topic_label': 'Financial Stability', 'confi..."
5,Financial Stability,The macroprudential stance is broadly appropri...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the macroprudential st...,"[{'topic_label': 'Financial Stability', 'confi..."
6,Monetary Policy,Durably reducing inflation to its target and a...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the need for a tight m...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
7,Monetary Policy,The authorities noted that the current monetar...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the current monetary p...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
8,Monetary Policy,The BSP’s prompt action to fight inflation is ...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the monetary policy ac...,"[{'topic_label': 'Monetary Policy', 'confidenc..."
9,Fiscal Stance,The authorities broadly agreed with the need f...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the need for medium-te...,"[{'topic_label': 'Fiscal Stance', 'confidence_..."


##### Check for output parsing errors 

In [200]:

output_fixing_pt = {'System':"You are an intelligent assistant specialized in formatting text. Your task is to take raw LLM output and format it according to user-provided instructions. Follow the specific formatting guidelines given and ensure the final output is clean, readable, and adheres to the specified style.",
      'Human':"""
Here is the raw LLM output:
----------------
----------------
{LLM_OUTPUT}
----------------
----------------

You are supposed to extract appropriate information for reasoning and topic_labels. topic_labels has additional information for confidence score.

Please respond in clean json format as follow:
```json
{{"reasoning": "<reasoning process>", 
"topic_labels": [{{"topic_label":"<identified topic label>","confidence_score":<confidence score>}},...]}}
```
Response:
"""}

In [201]:
def custom_llm_result_parsing(llm_res, llm_agent, json_parse=True,parser=None, output_fixing_pt_temp=None,verbose=False):
    def parse_result(res, parser):
        try:
            return parser.parse(res).dict()
        except Exception as e:
            if verbose:
                print(f"Parser error: {e}")
            return None

    if parser:
        res_dict = parse_result(llm_res, parser)
        if res_dict is None:
            if json_parse:
                try:
                    res_dict = llm_agent.parse_load_json_str(llm_res)
                    if verbose:
                        print('Use basic json parse to fix output ...')
                except Exception as e:
                    if verbose:
                        print(f"JSON parsing error: {e}")
                    res_dict = None
            if res_dict is None and output_fixing_pt_temp:
                try:
                    new_res = llm_agent.get_response_content(prompt_template=output_fixing_pt_temp, max_tokens=4096,temperature=0)
                    #print(res)
                    res_dict = custom_llm_result_parsing(new_res, llm_agent,json_parse,parser, False)
                    if verbose:
                        print('Use llm to fix output ...')
                except Exception as e:
                    if verbose:
                        print(f"Output fixing template error: {e}")
                    res_dict = None
        return res_dict
    else:
        return None

In [205]:
error_example = res_df[res_df['reasoning'].isna()].iloc[2]
#print(error_example['llm_raw_response'])
output_fixing_pt_temp = copy.copy(output_fixing_pt)
output_fixing_pt_temp['Human']=output_fixing_pt_temp['Human'].format(LLM_OUTPUT=error_example['llm_raw_response'])
r_dict = custom_llm_result_parsing(error_example['llm_raw_response'],llm_agent,json_parse=True,parser=parser,output_fixing_pt_temp=output_fixing_pt_temp)
print(r_dict)


{'reasoning': "The paragraph discusses the administration's views on trade policies, their goals, and the actions they are taking to achieve those goals. It mentions the importance of a fair international trading system, addressing the global climate crisis, respecting the dignity of work, and ensuring manufacturing supply chains are resilient. It also talks about the administration's commitment to executing 'Made in America Laws' consistent with existing international obligations. The focus is on trade policies, their impact on U.S. workers and communities, and the administration's efforts to address unfair practices by U.S. trading partners. The discussion of currency practices and procurement rules is also related to trade policies. Therefore, the topic that best fits the paragraph is **Economic Outlook**.", 'topic_labels': [{'topic_label': 'Economic Outlook', 'confidence_score': 0}]}


In [206]:
error_results = []
for idx,r in tqdm(res_df[res_df['reasoning'].isna()].iterrows()):
    output_fixing_pt_temp = copy.copy(output_fixing_pt)
    output_fixing_pt_temp['Human']=output_fixing_pt_temp['Human'].format(LLM_OUTPUT=r['llm_raw_response'])
    #res = llm_agent.get_response_content(prompt_template=output_fixing_pt_temp,max_tokens=4096)
    res_dict = custom_llm_result_parsing(r['llm_raw_response'], llm_agent,json_parse=True, parser=parser, output_fixing_pt_temp=output_fixing_pt_temp)
    #print(res_dict)
    if res_dict:
        error_results.append([r['ground_truth_label'],r['paragraph'],r['llm_raw_response'],res_dict['reasoning'],res_dict['topic_labels']])
    else:
        error_results.append([r['ground_truth_label'],r['paragraph'],r['llm_raw_response'],None,None])

3it [00:04,  1.52s/it]


In [207]:
error_res_df = pd.DataFrame(error_results,columns=['ground_truth_label','paragraph','llm_raw_response','reasoning','topic_labels'])
error_res_df.head(10)

,ground_truth_label,paragraph,llm_raw_response,reasoning,topic_labels
0,Governance,Box 6. Corruption and AML/CFT Reforms Over the...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses Mexico's efforts to co...,"[{'topic_label': 'Governance', 'confidence_sco..."
1,Other Topics,Better data are needed to support domestic pol...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the importance of data...,[{'topic_label': 'Data Provision and Dissemina...
2,External Stance,Authorities’ views. The administration is focu...,"```json\n{\n ""reasoning"": ""The paragraph disc...",The paragraph discusses the administration's v...,"[{'topic_label': 'Economic Outlook', 'confiden..."


In [38]:
res_df.to_csv('/root/data/Fund/CSR/llm_detailed_topic_results_llama-8b.csv')